##gold_customer_value

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.final.gold_customer_value AS

SELECT
    c.Customer_SK,
    c.CustomerID,
    c.customer_type,

    COUNT(DISTINCT f.SalesOrderID) AS OrderCount,

    ROUND(
        SUM(f.NetSalesAmount),
        2
    ) AS LifetimeRevenue,

    ROUND(
        SUM(
            f.NetSalesAmount -
            (
                f.OrderQty *
                p.StandardCost
            )
        ),
        2
    ) AS LifetimeProfit,

    ROUND(
        (
            SUM(
                f.NetSalesAmount -
                (
                    f.OrderQty *
                    p.StandardCost
                )
            )
            /
            SUM(f.NetSalesAmount)
        ) * 100,
        2
    ) AS ProfitMarginPct

FROM workspace.final.gold_fact_sales f

JOIN workspace.final.gold_dim_customer c
    ON f.Customer_SK = c.Customer_SK

JOIN workspace.final.gold_dim_product p
    ON f.Product_SK = p.Product_SK

GROUP BY
    c.Customer_SK,
    c.CustomerID,
    c.customer_type

##gold_product_profitability

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.final.gold_product_profitability AS

SELECT
    p.Product_SK,
    p.ProductID,
    p.ProductName,
    p.Color,

    SUM(f.OrderQty) AS TotalUnitsSold,

    ROUND(
        SUM(f.NetSalesAmount),
        2
    ) AS Revenue,

    ROUND(
        SUM(
            f.NetSalesAmount -
            (
                f.OrderQty *
                p.StandardCost
            )
        ),
        2
    ) AS Profit,

    ROUND(
        (
            SUM(
                f.NetSalesAmount -
                (
                    f.OrderQty *
                    p.StandardCost
                )
            )
            /
            SUM(f.NetSalesAmount)
        ) * 100,
        2
    ) AS ProfitMarginPct

FROM workspace.final.gold_fact_sales f

JOIN workspace.final.gold_dim_product p
    ON f.Product_SK = p.Product_SK

GROUP BY
    p.Product_SK,
    p.ProductID,
    p.ProductName,
    p.Color

##gold_sales_trend

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.final.gold_sales_trend AS

SELECT
    d.Year,
    d.Quarter,
    d.Month,
    d.MonthName,

    COUNT(DISTINCT f.SalesOrderID) AS Orders,

    ROUND(
        SUM(f.NetSalesAmount),
        2
    ) AS Revenue,

    ROUND(
        SUM(
            f.NetSalesAmount -
            (
                f.OrderQty *
                p.StandardCost
            )
        ),
        2
    ) AS Profit

FROM workspace.final.gold_fact_sales f

JOIN workspace.final.gold_dim_date d
    ON f.OrderDate_SK = d.Date_SK

JOIN workspace.final.gold_dim_product p
    ON f.Product_SK = p.Product_SK

GROUP BY
    d.Year,
    d.Quarter,
    d.Month,
    d.MonthName

##gold_customer_type_performance

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.final.gold_customer_type_performance AS

SELECT
    c.customer_type,

    COUNT(DISTINCT c.CustomerID) AS Customers,

    ROUND(
        SUM(f.NetSalesAmount),
        2
    ) AS Revenue,

    ROUND(
        SUM(
            f.NetSalesAmount -
            (
                f.OrderQty *
                p.StandardCost
            )
        ),
        2
    ) AS Profit,

    ROUND(
        (
            SUM(
                f.NetSalesAmount -
                (
                    f.OrderQty *
                    p.StandardCost
                )
            )
            /
            SUM(f.NetSalesAmount)
        ) * 100,
        2
    ) AS ProfitMarginPct

FROM workspace.final.gold_fact_sales f

JOIN workspace.final.gold_dim_customer c
    ON f.Customer_SK = c.Customer_SK

JOIN workspace.final.gold_dim_product p
    ON f.Product_SK = p.Product_SK

GROUP BY c.customer_type

##gold_customer_acquisition

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.final.gold_customer_acquisition
USING DELTA
AS

WITH first_purchase AS
(
    SELECT
        Customer_SK,

        MIN(OrderDate_SK)
        AS FirstOrderDate

    FROM workspace.final.gold_fact_sales

    GROUP BY Customer_SK
)

SELECT
    d.Year,
    d.Month,
    d.MonthName,

    COUNT(*)
    AS NewCustomers

FROM first_purchase fp

JOIN workspace.final.gold_dim_date d
ON fp.FirstOrderDate = d.Date_SK

GROUP BY
    d.Year,
    d.Month,
    d.MonthName